# 06 — Window Leakage Analysis
### Mục 0.3 (phần 1) — Rò rỉ dữ liệu do Sliding Window

**Mục đích**: minh họa bằng số **bản chất** của rò rỉ dữ liệu khi chia
ngẫu nhiên theo sliding window, và chạy thực nghiệm nhỏ đối chiếu Random
Window Split vs File-based Split — bằng chứng thực nghiệm **sơ bộ** cho
RQ1/H1 (kết quả đầy đủ, chính thức vẫn báo cáo lại ở Giai đoạn 4 với bể
đặc trưng đầy đủ và LOLO 4-fold).

**Lưu ý phương pháp quan trọng**: minh họa dưới đây đo **tỉ lệ mẫu thô
thực sự trùng nhau** giữa 2 window liền kề (một sự thật xác định) và
**chênh lệch đặc trưng thống kê** giữa chúng — KHÔNG dùng hệ số tương quan
Pearson so theo vị trí chỉ số (`corrcoef(w1, w2)`), vì cách đó có thể cho
số liệu sai lệch/gây hiểu nhầm khi tín hiệu có thành phần tuần hoàn lệch
pha ngẫu nhiên với bước dịch cửa sổ.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from common import io_utils, pipeline, features, splitting
pd.set_option("display.max_colwidth", 120)

In [2]:
# ============================== CẤU HÌNH ==============================
# Đổi USE_SYNTHETIC_DATA = False và chỉnh REAL_DATA_ROOT khi đã có dữ liệu
# CWRU thật. Xem README.md phần "Chuyển sang dữ liệu thật".
USE_SYNTHETIC_DATA = True
REAL_DATA_ROOT = Path("../../data/raw")            # <-- de_tai_nckh/data/raw/
SYNTHETIC_DATA_ROOT = Path("./_data/synthetic_cwru")
OUTPUT_DIR = Path("./outputs")
FORCE_REBUILD_MANIFEST = False
# ========================================================================

In [3]:
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

> ⚠️ **Lưu ý dữ liệu giả lập**: khi `USE_SYNTHETIC_DATA = True`, mọi tín hiệu
> trong notebook này là **giả lập** (nhiễu + xung điều biên mô phỏng), chỉ để
> kiểm tra code chạy đúng và xem trước hình dạng đầu ra. **Không dùng số liệu
> giả lập này làm kết quả báo cáo chính thức.** Khi có dữ liệu CWRU thật, đổi
> `USE_SYNTHETIC_DATA = False` ở cell CẤU HÌNH và chạy lại toàn bộ notebook.

In [4]:
manifest = pipeline.get_manifest(
    use_synthetic=USE_SYNTHETIC_DATA,
    real_data_root=REAL_DATA_ROOT,
    synthetic_data_root=SYNTHETIC_DATA_ROOT,
    output_dir=OUTPUT_DIR,
    force_rebuild=FORCE_REBUILD_MANIFEST,
)
print(f"Tổng số file trong manifest: {len(manifest)}")
manifest.head()

Tổng số file trong manifest: 40


,file_path,load_hp,label,fault_diameter_mils,or_position,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning
0,_data/synthetic_cwru/0hp/B_007.mat,0,B,7.0,NaN,120000,NaN,NaN,1797.0,NaN,NaN,False
1,_data/synthetic_cwru/0hp/B_014.mat,0,B,14.0,NaN,120000,NaN,NaN,1797.0,NaN,NaN,False
2,_data/synthetic_cwru/0hp/B_021.mat,0,B,21.0,NaN,120000,NaN,NaN,1797.0,NaN,NaN,False
3,_data/synthetic_cwru/0hp/IR_007.mat,0,IR,7.0,NaN,120000,NaN,NaN,1797.0,NaN,NaN,False
4,_data/synthetic_cwru/0hp/IR_014.mat,0,IR,14.0,NaN,120000,NaN,NaN,1797.0,NaN,NaN,False


## Minh họa 1 — Tỉ lệ mẫu trùng & chênh lệch đặc trưng giữa 2 window liền kề

In [5]:
def demo_window_overlap_similarity(signal, window_size, overlap_ratio):
    step = max(int(window_size * (1 - overlap_ratio)), 1)
    w1 = signal[0: window_size]
    w2 = signal[step: step + window_size]

    shared_fraction = max(0, window_size - step) / window_size
    f1 = features.extract_simple_features(w1)
    f2 = features.extract_simple_features(w2)

    print(f"Overlap cấu hình = {overlap_ratio*100:.0f}%  (step={step}, window={window_size})")
    print(f"  -> Tỉ lệ MẪU THÔ trùng nhau: {shared_fraction*100:.1f}%")
    print("  -> Chênh lệch đặc trưng thống kê giữa 2 window:")
    for name in f1:
        rel_diff = abs(f1[name] - f2[name]) / (abs(f1[name]) + 1e-8) * 100
        print(f"       {name:10s}: {rel_diff:6.2f}% chênh lệch")
    print()
    return shared_fraction

fp = pipeline.pick_file(manifest, label="OR", load_hp=0)
demo_signal = io_utils.load_de_signal(Path(fp))

for overlap in (0.9, 0.75, 0.5, 0.0):
    demo_window_overlap_similarity(demo_signal, window_size=2048, overlap_ratio=overlap)

Overlap cấu hình = 90%  (step=204, window=2048)
  -> Tỉ lệ MẪU THÔ trùng nhau: 90.0%
  -> Chênh lệch đặc trưng thống kê giữa 2 window:
       rms       :   0.58% chênh lệch
       kurtosis  :  23.27% chênh lệch
       skewness  :   5.30% chênh lệch
       peak      :   0.00% chênh lệch
       std       :   0.54% chênh lệch

Overlap cấu hình = 75%  (step=512, window=2048)
  -> Tỉ lệ MẪU THÔ trùng nhau: 75.0%
  -> Chênh lệch đặc trưng thống kê giữa 2 window:
       rms       :   0.26% chênh lệch
       kurtosis  :  21.89% chênh lệch
       skewness  :  29.54% chênh lệch
       peak      :   0.00% chênh lệch
       std       :   0.25% chênh lệch

Overlap cấu hình = 50%  (step=1024, window=2048)
  -> Tỉ lệ MẪU THÔ trùng nhau: 50.0%
  -> Chênh lệch đặc trưng thống kê giữa 2 window:
       rms       :   0.73% chênh lệch
       kurtosis  :  35.11% chênh lệch
       skewness  :   2.67% chênh lệch
       peak      :   9.50% chênh lệch
       std       :   0.79% chênh lệch

Overlap cấu hình = 0%

## Minh họa 2 — Thực nghiệm so sánh Random Window Split vs File-based Split

Cắt sliding window (overlap 75%) trên **toàn bộ** file trong manifest,
trích 5 đặc trưng đơn giản, huấn luyện Random Forest theo 2 cách chia,
so sánh accuracy.

In [6]:
WINDOW_SIZE = 2048
OVERLAP_RATIO = 0.75

all_windows = []
for _, row in manifest.iterrows():
    if row["label"] is None or pd.isna(row["label"]):
        continue
    x = io_utils.load_de_signal(Path(row["file_path"]))
    file_id = f"{row['label']}_{row['load_hp']}_{row.get('fault_diameter_mils')}_{row['file_path']}"
    w = features.make_sliding_windows(x, WINDOW_SIZE, OVERLAP_RATIO, file_id=file_id, label=row["label"])
    all_windows.append(w)

windows_df = pd.concat(all_windows, ignore_index=True)
feature_df = features.build_feature_table(windows_df)
feature_cols = ["rms", "kurtosis", "skewness", "peak", "std"]
print(f"Tổng số window: {len(feature_df)}  (từ {feature_df['file_id'].nunique()} file)")

Tổng số window: 9240  (từ 40 file)


In [7]:
result_df = splitting.run_split_comparison_experiment(feature_df, feature_cols)
result_df.to_csv(TABLES_DIR / "06_split_comparison.csv")

delta = result_df.loc["Random Window Split (SAI)", "accuracy"] - \
        result_df.loc["File-based Split (ĐÚNG)", "accuracy"]

print(result_df)
print(f"\nΔAccuracy (ảo) = {delta:.4f}")
print("Dương và đáng kể -> khớp chiều với H1: Random Window Split cho "
      "accuracy cao hơn ảo do rò rỉ dữ liệu.")

                           accuracy  f1_macro  n_train  n_test
Random Window Split (SAI)  0.981602  0.984513   6468.0  2772.0
File-based Split (ĐÚNG)    0.976190  0.961810   6468.0  2772.0

ΔAccuracy (ảo) = 0.0054
Dương và đáng kể -> khớp chiều với H1: Random Window Split cho accuracy cao hơn ảo do rò rỉ dữ liệu.


## Quan sát & kết luận sơ bộ

- Ghi lại ΔAccuracy quan sát được — đây là bằng chứng **sơ bộ** cho RQ1/H1.
- Kết quả **chính thức** để đưa vào Bảng 1 của báo cáo (mục 4.1) phải chạy
  lại ở Giai đoạn 4 với: (a) bể đặc trưng đầy đủ (mục 1.4, không chỉ 5 đặc
  trưng đơn giản ở đây), và (b) File-based Split + LOLO đầy đủ 4-fold
  (không chỉ 1 lần chia ngẫu nhiên như demo này).

> ⚠️ **Vì sao ΔAccuracy có thể RẤT NHỎ khi chạy với `USE_SYNTHETIC_DATA =
> True`?** Tín hiệu giả lập trong `common/synthetic.py` tạo ra các lớp
> tách biệt rất rõ ràng (mỗi lỗi có 1 tần số xung riêng biệt, không nhiễu
> chồng lấp giữa các lớp) — cả 2 cách chia đều dễ dàng đạt gần 100%
> accuracy ("hiệu ứng trần"/ceiling effect), khiến chênh lệch do rò rỉ dữ
> liệu bị che khuất dù nó vẫn tồn tại về bản chất (xem lại Minh họa 1 ở
> trên — tỉ lệ mẫu trùng và chênh lệch đặc trưng vẫn rất rõ). **Đây KHÔNG
> phải bằng chứng phản bác H1.** Với dữ liệu CWRU thật (nhiễu cơ khí thật,
> ranh giới giữa các lớp mờ hơn nhiều), y văn báo cáo chênh lệch do rò rỉ
> thường rất đáng kể (accuracy ảo >99% so với accuracy tổng quát hóa thấp
> hơn nhiều) — đây chính là động lực ban đầu của đề tài (xem mục 0 —
> Bối cảnh và động lực nghiên cứu). Chạy lại notebook này với dữ liệu thật
> để có con số đáng tin cậy.

Bảng kết quả đã lưu tại `outputs/tables/06_split_comparison.csv`.